In [ ]:
# Settings

from pathlib import Path, PureWindowsPath
import pandas as pd
import numpy as np
from datetime import datetime

latMeter = 0.0000089988659514815        # 1 meter expressed in latitude (works for area province Utrecht)
lonMeter = 0.0000146436902975532        # 1 meter expressed in longitude (works for area province Utrecht)
latMin = 0 #51.858631                      # smallest latitude province Utrecht
lonMin = 0 #4.794457                       # smallest longitude province Utrecht
distance = 10                           # distance (meters) from route to attribute to route(segment)
isExtend = True
timestamp = str(int(datetime.timestamp(datetime.now())))
prefix = "mcu_gegevens"

#TEST routes
routes = ('centraal-koningsweg-usp.csv', 
    'centraal-wilhelminapark-usp.csv', 
    'centraal-biltstraat-usp.csv')

#SET FOLDER TO LOCATE mcu_gegevens_2024-09.csv
data_directory = PureWindowsPath('D:\\Snuffelfietsdata\\mcu_bewerkt')

#SET FOLDER TO LOCATE TEST routes
output_directory = Path(
    Path('~').expanduser(),
    'Documents',
    'MCUdataclub',
    'Snuffelfiets_data',
    'route-filter',
    )
Path.mkdir(output_directory, parents=True, exist_ok=True)

print(f'writing output to {output_directory}.')


writing output to C:\Users\Work\Documents\MCUdataclub\Snuffelfiets_data\route-filter.


In [2]:
# test: read all data from D:\Snuffelfietsdata\mcu_bewerkt
dfs = []

for year in [2024]:                                         #[2021, 2022, 2023, 2024]
    for month in [9]:   #[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]:
        filename = f'{prefix}_{year}-{month:02d}.csv'
        p = Path(data_directory, filename)
        df = pd.read_csv(p)
        dfs.append(df)

df = pd.concat(dfs, ignore_index=True) #bug fixed by ignore_index=True

# add columns yLat and xLon to df
df['yLat'] = (df['latitude'] - latMin) / latMeter
df['xLon'] = (df['longitude'] - lonMin) / lonMeter

# make rit_id unique
df['rit_id'] = df['rit_id'] + df['year'] * 1000000 + df['month'] * 10000

print(f'Read {df.shape[0]} measurements.')

Read 188934 measurements.


In [3]:
# Import route points
for filename in routes:
    p = Path(output_directory, filename)
    dfR = pd.read_csv(p)
    print(f'Read {dfR.shape[0]} route points.')

    # Add columns xB and yB to dfR
    dfR['xB'] = (dfR['longitude'] - lonMin) / lonMeter
    dfR['yB'] = (dfR['latitude'] - latMin) / latMeter
    dfR['xE'] = dfR['xB'].shift(-1)
    dfR['yE'] = dfR['yB'].shift(-1)
    dfR = dfR[:-1]
    dfR['BE'] = np.sqrt( 
        np.square(dfR['xE'] - dfR['xB']) 
        + np.square(dfR['yE'] - dfR['yB']) )

    # Extend segment from end point E with set distance
    if isExtend: 
        dfR['xE'] = (dfR['xE'] - dfR['xB']) * (distance / dfR['BE']) + dfR['xE'] 
        dfR['yE'] = (dfR['yE'] - dfR['yB']) * (distance / dfR['BE']) + dfR['yE'] 
        dfR['BE'] = np.sqrt( 
            np.square(dfR['xE'] - dfR['xB']) 
            + np.square(dfR['yE'] - dfR['yB']) ) 

    dfs = []

    #Loop through route segments
    for index, segment in dfR.iterrows():

        # Calculate sides MB, ME using Pythagorean theorem
        df['MB'] = np.sqrt( 
            np.square(df['xLon'] - segment['xB']) 
            + np.square(df['yLat'] - segment['yB']) ) 

        df['ME'] = np.sqrt( 
            np.square(df['xLon'] - segment['xE']) 
            + np.square(df['yLat'] - segment['yE']) ) 

        # Calculate angles aBE, aMB, aME using Law of cosines
        # TODO: check if calculating angle aBE is redundant
        df['aBE'] = np.acos( 
            ( np.square(df['MB']) + np.square(df['ME']) - np.square(segment['BE']) ) 
            / (2 * df['MB'] * df['ME']) ) 

        df['aMB'] = np.acos( 
            ( np.square(df['ME']) + np.square(segment['BE']) - np.square(df['MB']) ) 
            / (2 * segment['BE'] * df['ME']) ) 

        df['aME'] = np.acos( 
            ( np.square(df['MB']) + np.square(segment['BE']) - np.square(df['ME']) ) 
            / (2 * segment['BE'] * df['MB']) ) 

        # Calculate distance M-BE from measurement perpendicular to route segment line BE
        df['M-BE'] = np.sin(df['aME']) * df['MB'] 

        # Filter measurements to temporary dataframe dfT
        dfT = df[df['M-BE'].between(0, distance)]           #M-BE can't be more than set distance
        dfT = dfT[dfT['aMB'].between(0, np.pi/2)]           #Angle aMB can't be more than 90 degrees
        dfT = dfT[dfT['aME'].between(0, np.pi/2)]           #Angle aME can't be more than 90 degrees

        # Add marker
        dfT['routeSegment'] = index + 1

        # Add temp segment df (dfT) to output df
        dfs.append(dfT)

        # Clear temp segment df
        del dfT

    dfOutput = pd.concat(dfs, ignore_index=True) #bug fixed by ignore_index=True

    # export dfOutput
    filename2 = 'dfOutput-' + timestamp  + "-" + filename
    p = Path(output_directory, filename2)
    dfOutput.to_csv(p, index=False)

Read 189 route points.


c:\Users\Work\miniforge3\envs\snuffelfiets\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Work\miniforge3\envs\snuffelfiets\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Work\miniforge3\envs\snuffelfiets\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Work\miniforge3\envs\snuffelfiets\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Work\miniforge3\envs\snuffelfiets\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Work\minifo

Read 193 route points.


c:\Users\Work\miniforge3\envs\snuffelfiets\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Work\miniforge3\envs\snuffelfiets\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Work\miniforge3\envs\snuffelfiets\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Work\miniforge3\envs\snuffelfiets\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Work\miniforge3\envs\snuffelfiets\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Work\minifo

Read 174 route points.


c:\Users\Work\miniforge3\envs\snuffelfiets\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Work\miniforge3\envs\snuffelfiets\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Work\miniforge3\envs\snuffelfiets\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Work\miniforge3\envs\snuffelfiets\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Work\miniforge3\envs\snuffelfiets\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Work\minifo